# Intro to Archiving

mt_sb_archiving is a Python-based utility library designed for uploading magnetotelluric (MT) data files to USGS's ScienceBase database for archival. In general, we prefer to use HDF5 files, so that we can use the [MTH5 library](https://github.com/IAGA-DVI-DataStandards/mth5) to store data, which is then encoded in a JSON table. We also use XML to store metadata and transfer functions.

## Creating XMLs

This section will focus on reducing MTH5 files to XMLs so that they can readily be uploaded to the relevant pages on ScienceBase. The aim here is to not modify any of the MTH5s present within the directory we wish to upload, nor have to rebuild them over the course of this process.

Credit to Jade Crosbie (jcrosbie@usgs.gov) for the algorithm.

In [ ]:
import sys
import mt_sb_archiving

from pathlib import Path
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

from mth5.mth5 import MTH5
from archive import mt_xml

In [ ]:
# folder containing the finished .h5 files (searched recursively)
SURVEY_DIR = Path(r"path/to/survey")

# where the parent XML, summary CSV, and shapefile go
OUTPUT_DIR = Path(r"path/to/outputs")

SB_DIR = Path(r"path/to/cfg/and/templates")
XML_CFG_FN = SB_DIR / "ghamsp1_xml_configuration.cfg"
XML_ROOT_TEMPLATE = SB_DIR / "mt_root_template.xml"
XML_CHILD_TEMPLATE = SB_DIR / "mt_child_template.xml"

MAKE_CHILD_XML = True
MAKE_PARENT_XML = True

# values put in the shapefile / summary that aren't stored in the MTH5
DATA_TYPE = "WBMT"
QUALITY = 0 # just the default if missing!

# folder names to skip when searching for .h5 files
SKIP_DIRS = {"Archive"}

# Keyword text that is template placeholder/instruction text, not a real term.
# Compared case-insensitively with whitespace collapsed. Add to this if other
# placeholders show up. Keyword blocks left with no keys are removed.
PLACEHOLDER_KEYWORDS = {
    "place names",
    "states, counties",
    "states",
    "counties",
    "geologic terranes",
    "ages in the geolexicon data base https://ngmdb.usgs.gov/geolex/search",
    "ages not in the geolexicon",
    "none",
}

The above cells will handle all of the variables and imports we will need to make this happen (you can fill in some of the placeholder values, such as the ones for SURVEY_DIR and OUTPUT_DIR, for your own needs). We will also define a handful of helper functions as well:

In [ ]:
def to_timestamp(value):
    """Convert an MTime / string to a timezone-naive pandas Timestamp."""
    try:
        ts = pd.Timestamp(str(value))
    except Exception:
        return pd.NaT
    if ts.tzinfo is not None:
        ts = ts.tz_convert("UTC").tz_localize(None)
    return ts


def safe_get(func, default=""):
    """Return func() or a default value if anything goes wrong."""
    try:
        value = func()
        return default if value is None else value
    except Exception:
        return default


def _norm(text):
    """Lower-case and collapse whitespace for keyword comparison."""
    return " ".join((text or "").split()).lower()


def clean_xml(xml_fn, station=None):
    """
    Post-process a saved FGDC XML file.

    * If ``station`` is given, append ", station <station>" to the main
      citation title (idinfo/citation/citeinfo/title). The larger-work
      citation (lworkcit) that points to the parent is left alone.
    * Remove placeholder keywords and any keyword block left empty.
    """
    tree = ET.parse(xml_fn)
    root = tree.getroot()

    if station is not None:
        title = root.find("idinfo/citation/citeinfo/title")
        if title is not None and title.text:
            suffix = f", station {station}"
            base = title.text.strip()
            if not base.endswith(suffix):
                title.text = base + suffix
        else:
            print(f"  WARNING: no citation title found in {xml_fn}")

    keywords = root.find("idinfo/keywords")
    if keywords is not None:
        for block_tag, key_tag in (
            ("theme", "themekey"),
            ("place", "placekey"),
            ("stratum", "stratkey"),
            ("temporal", "tempkey"),
        ):
            for block in keywords.findall(block_tag):
                for key in block.findall(key_tag):
                    if _norm(key.text) in PLACEHOLDER_KEYWORDS or not _norm(key.text):
                        block.remove(key)
                if not block.findall(key_tag):
                    keywords.remove(block)
    # keep only the dates in the time period; drop begtime/endtime
    rngdates = root.find("idinfo/timeperd/timeinfo/rngdates")
    if rngdates is not None:
        for tag in ("begtime", "endtime"):
            for el in rngdates.findall(tag):
                rngdates.remove(el)
    ET.indent(tree, space="  ")
    tree.write(xml_fn, encoding="UTF-8", xml_declaration=True)
    return xml_fn


def iter_station_groups(m):
    """Yield station groups for both MTH5 file version 0.1.0 and 0.2.0."""
    if m.file_version == "0.1.0":
        stations_group = m.stations_group
        for name in stations_group.groups_list:
            yield stations_group.get_station(name)
    else:
        for survey_name in m.surveys_group.groups_list:
            survey = m.surveys_group.get_survey(survey_name)
            for name in survey.stations_group.groups_list:
                yield survey.stations_group.get_station(name)


def iter_runs(station_group):
    """Yield run groups, skipping Transfer_Functions / Fourier_Coefficients."""
    for name in station_group.groups_list:
        if name in ("Transfer_Functions", "Fourier_Coefficients"):
            continue
        try:
            yield station_group.get_run(name)
        except Exception:
            continue

def get_quality(station_group, default=QUALITY):
    """
    Return the TF data-quality rating stored in this station's
    Transfer_Functions group, or `default` if none is found.
    """
    try:
        tf_group = station_group.transfer_functions_group
    except Exception:
        return default

    for tf_id in tf_group.groups_list:
        try:
            tmeta = tf_group.get_transfer_function(tf_id).metadata
        except Exception:
            continue
        # the TF metadata can be the TransferFunction object itself or
        # wrapped in a station object, depending on version
        dq = getattr(tmeta, "data_quality", None)
        if dq is None:
            dq = safe_get(lambda: tmeta.transfer_function.data_quality, None)
        value = safe_get(lambda: dq.rating.value, None)
        if value is not None:
            return int(value)
    return default

def summarize_mth5(h5_fn):
    """
    Read one MTH5 file and return a list of station summary dictionaries
    (normally one per file).
    """
    entries = []
    m = MTH5()
    m.open_mth5(h5_fn, "r")
    try:
        for sg in iter_station_groups(m):
            smeta = sg.metadata
            entry = {
                "station": smeta.id,
                "h5_fn": str(h5_fn),
                "latitude": safe_get(lambda: smeta.location.latitude, np.nan),
                "longitude": safe_get(lambda: smeta.location.longitude, np.nan),
                "elevation": safe_get(lambda: smeta.location.elevation, np.nan),
                "operator": safe_get(lambda: smeta.acquired_by.author),
                "type": DATA_TYPE,
                "quality": get_quality(sg),
            }

            starts, ends, sample_rates, loggers = [], [], set(), []
            components = set()
            ch_info = {}

            for rg in iter_runs(sg):
                rmeta = rg.metadata
                sample_rates.add(safe_get(lambda: rmeta.sample_rate, np.nan))
                loggers.append(safe_get(lambda: rmeta.data_logger.id))

                for ch_name in rg.groups_list:
                    try:
                        ch = rg.get_channel(ch_name)
                    except Exception:
                        continue
                    cmeta = ch.metadata
                    comp = cmeta.component.lower()
                    components.add(comp)
                    starts.append(to_timestamp(cmeta.time_period.start))
                    ends.append(to_timestamp(cmeta.time_period.end))

                    # keep the first value found for each component
                    if comp in ch_info:
                        continue
                    info = {
                        "azimuth": safe_get(lambda: cmeta.measurement_azimuth, np.nan)
                    }
                    if comp.startswith("e"):
                        info["length"] = safe_get(lambda: cmeta.dipole_length, np.nan)
                    elif comp.startswith("h"):
                        info["sensor"] = safe_get(lambda: cmeta.sensor.id)
                        info["tilt"] = safe_get(lambda: cmeta.measurement_tilt, np.nan)
                    ch_info[comp] = info

            entry["start"] = min([s for s in starts if pd.notna(s)], default=pd.NaT)
            entry["end"] = max([e for e in ends if pd.notna(e)], default=pd.NaT)
            entry["components"] = ",".join(sorted(components))
            entry["n_chan"] = len(components)
            entry["n_runs"] = len(list(iter_runs(sg)))
            entry["sample_rates"] = ",".join(
                f"{sr:g}" for sr in sorted(sample_rates) if pd.notna(sr)
            )
            loggers = [l for l in loggers if l]
            entry["data_logger"] = max(set(loggers), key=loggers.count) if loggers else ""

            for ec in ["ex", "ey"]:
                entry[f"{ec}_length"] = ch_info.get(ec, {}).get("length", np.nan)
                entry[f"{ec}_azimuth"] = ch_info.get(ec, {}).get("azimuth", np.nan)
            for hc in ["hx", "hy", "hz"]:
                entry[f"{hc}_sensor"] = str(ch_info.get(hc, {}).get("sensor", ""))
                entry[f"{hc}_azimuth"] = ch_info.get(hc, {}).get("azimuth", np.nan)
            # the shapefile uses hz inclination, which is the tilt for hz
            entry["hz_azimuth"] = ch_info.get("hz", {}).get(
                "tilt", entry["hz_azimuth"]
            )

            entries.append(entry)
    finally:
        m.close_mth5()
    return entries


def make_child_xml(entry, save_dir):
    """Write <station>.xml for one station summary entry."""
    station = entry["station"]
    s_xml = mt_xml.MTSBXML()
    s_xml.read_template_xml(XML_CHILD_TEMPLATE)
    s_xml.update_from_config(XML_CFG_FN, child=True)
    s_xml.update_with_station(station)

    # west, east, north, south
    s_xml.update_bounding_box(
        entry["longitude"], entry["longitude"], entry["latitude"], entry["latitude"]
    )
    s_xml.update_time_period(entry["start"].isoformat(), entry["end"].isoformat())
    s_xml.update_metadate()

    xml_fn = Path(save_dir).joinpath(f"{station}.xml")
    s_xml.save(xml_fn)

    # station-specific title + remove placeholder keywords
    clean_xml(xml_fn, station=station)
    return xml_fn


def write_shp_file(survey_df, save_fn):
    """Write the survey shapefile with the same columns archive.py uses."""
    shp_df = survey_df.rename(
        columns={
            "data_logger": "instr_id",
            "station": "siteID",
            "hz_azimuth": "hz_inclina",
            "start": "start_date",
            "end": "end_date",
        }
    )
    shp_df["sitePrefix"] = shp_df["siteID"].str[:2]
    for col in ["start_date", "end_date"]:
        shp_df[col] = shp_df[col].astype(str)

    col_list = [
        "sitePrefix","siteID", "latitude", "longitude", "elevation",
        "hx_azimuth", "hy_azimuth", "hz_inclina",
        "hx_sensor", "hy_sensor", "hz_sensor",
        "ex_length", "ey_length", "ex_azimuth", "ey_azimuth",
        "n_chan", "instr_id", "operator", "type", "quality",
        "start_date", "end_date",
    ]
    shp_df = shp_df[col_list]
    geometry = [Point(x, y) for x, y in zip(shp_df.longitude, shp_df.latitude)]
    gdf = gpd.GeoDataFrame(shp_df, crs="EPSG:4326", geometry=geometry)
    gdf.to_file(save_fn)
    return shp_df


def make_parent_xml(survey_df, shp_df, save_fn):
    """Write the parent page XML from the survey summary."""
    p_xml = mt_xml.MTSBXML()
    p_xml.read_template_xml(XML_ROOT_TEMPLATE)
    p_xml.update_from_config(XML_CFG_FN)

    # west, east, north, south
    p_xml.update_bounding_box(
        survey_df.longitude.min(),
        survey_df.longitude.max(),
        survey_df.latitude.max(),
        survey_df.latitude.min(),
    )
    p_xml.update_time_period(
        survey_df.start.min().isoformat(), survey_df.end.max().isoformat()
    )
    p_xml.update_shp_attributes(shp_df)
    p_xml.update_metadate()
    p_xml.save(save_fn)

    # remove placeholder keywords (title stays the survey title)
    clean_xml(save_fn)
    return save_fn

Our main function will then make XMLs for both the parent page (housing the survey) and its children (housing the individual stations and the data they gathered)... 

In [ ]:
h5_list = sorted(
    fn
    for fn in SURVEY_DIR.rglob("*.h5")
    if not SKIP_DIRS.intersection(fn.relative_to(SURVEY_DIR).parts)
)
if not h5_list:
    raise IOError(f"No .h5 files found in {SURVEY_DIR}")
print(f"Found {len(h5_list)} MTH5 files")

all_entries = []
for h5_fn in h5_list:
    try:
        entries = summarize_mth5(h5_fn)
    except Exception as error:
        print(f"Skipping {h5_fn.name}: could not read ({error})")
        continue

    for entry in entries:
        all_entries.append(entry)
        if MAKE_CHILD_XML:
            try:
                xml_fn = make_child_xml(entry, h5_fn.parent)
                print(f"Wrote {xml_fn}")
            except Exception as error:
                print(f"Could not write XML for {entry['station']}: {error}")

survey_df = pd.DataFrame(all_entries)
survey_df.start = pd.to_datetime(survey_df.start)
survey_df.end = pd.to_datetime(survey_df.end)

# flag stations with missing location or times
bad = survey_df[
    survey_df[["latitude", "longitude", "start", "end"]].isna().any(axis=1)
    | (survey_df.latitude == 0)
    | (survey_df.longitude == 0)
]
if len(bad):
    print("\nWARNING: check location/time metadata for:")
    print(bad[["station", "latitude", "longitude", "start", "end"]])

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
csv_fn = OUTPUT_DIR.joinpath("survey_summary.csv")
survey_df.to_csv(csv_fn, index=False)
print(f"\nWrote {csv_fn}")

if MAKE_PARENT_XML:
    shp_fn = OUTPUT_DIR.joinpath("GHAMS_2025_site_locations.shp")
    shp_df = write_shp_file(survey_df, shp_fn)
    print(f"Wrote {shp_fn}")

    parent_fn = make_parent_xml(
        survey_df, shp_df, OUTPUT_DIR.joinpath("GHAMS_2025.xml")
    )
    print(f"Wrote {parent_fn}")

## Uploading to ScienceBase

With the XMLs organized with the relevant MTH5 files, we can upload these to ScienceBase now. You can look at the other notebook in this examples section to see how to do that.